In [1]:
import pandas as pd
import json
import os

In [2]:
# Читаем файлы напрямую с помощью pandas
portfolio = pd.read_json('../data/raw/portfolio.json', lines=True)
profile = pd.read_json('../data/raw/profile.json', lines=True)
transcript = pd.read_json('../data/raw/transcript.json', lines=True)

# Выводим размеры таблиц (сколько строк, сколько колонок)
print("portfolio:", portfolio.shape)
print("profile:", profile.shape)
print("transcript:", transcript.shape)

portfolio: (10, 6)
profile: (17000, 5)
transcript: (306534, 4)


In [3]:
# Смотрим на первые 3 строки каждой таблицы
print("--- ПОРТФОЛИО (ОФФЕРЫ) ---")
display(portfolio.head(3))

print("\n--- ПРОФИЛИ ПОЛЬЗОВАТЕЛЕЙ ---")
display(profile.head(3))

print("\n--- ТРАНСКРИПТ (СОБЫТИЯ) ---")
display(transcript.head(3))

--- ПОРТФОЛИО (ОФФЕРЫ) ---


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed



--- ПРОФИЛИ ПОЛЬЗОВАТЕЛЕЙ ---


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN



--- ТРАНСКРИПТ (СОБЫТИЯ) ---


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0


In [4]:
# 1. Сначала разберемся с профилями (profile)
# Заменим возраст 118 на NaN (пропуск), так как это явная ошибка/заглушка
profile['age'] = profile['age'].replace(118, pd.NA)

# Удалим пользователей без пола или без дохода (их не так много, они будут мешать анализу)
# Для этого оставим только те строки, где есть пол и доход
profile_clean = profile.dropna(subset=['gender', 'income']).copy()

print("Пользователей ДО чистки:", len(profile))
print("Пользователей ПОСЛЕ чистки:", len(profile_clean))

# Переведем дату регистрации из формата 20170212 в нормальную дату

profile_clean['became_member_on'] = pd.to_datetime(profile_clean['became_member_on'].astype(str), format='%Y%m%d')

Пользователей ДО чистки: 17000
Пользователей ПОСЛЕ чистки: 14825


In [5]:
# Создаем новую колонку offer_id и заполняем её
# Мы идем по каждой строке и смотрим, есть ли в словаре ключ 'offer id'
transcript['offer_id'] = transcript['value'].apply(lambda x: x.get('offer id') if 'offer id' in x else x.get('offer_id'))

# Создаем колонку amount для транзакций
transcript['amount'] = transcript['value'].apply(lambda x: x.get('amount') if 'amount' in x else None)

# Теперь колонка value нам больше не нужна, удалим её, чтобы не мешалась
transcript_clean = transcript.drop(columns=['value'])

# Смотрим, что получилось
display(transcript_clean.head(5))

,person,event,time,offer_id,amount
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN


In [6]:
# 1.  1. Создаем историческую переменную: совершал ли пользователь транзакции ВООБЩЕ (за все время)
# Для простоты берем всех, у кого была транзакция (event == 'transaction')
transactions = transcript_clean[transcript_clean['event'] == 'transaction'].groupby('person')['amount'].sum().reset_index()
transactions['any_purchase_history'] = 1

# 2. Мержим с профилем (оставляем только тех, кто есть в profile_clean)
df_model = profile_clean.merge(transactions[['person', 'any_purchase_history']], left_on='id', right_on='person', how='left')
# Если покупки не было, ставим 0 вместо NaN
df_model['any_purchase_history'] = df_model['any_purchase_history'].fillna(0).astype(int)

# 3. Создаем возрастные группы (как в KKBox)
df_model['age_group'] = pd.cut(df_model['age'], bins=[0, 25, 35, 50, 100], labels=['<25', '25-35', '35-50', '50+'])

# 4. Проверяем, что получилось
print(df_model[['gender', 'age', 'income', 'any_purchase_history']].head())
print("\nДоля покупок в целом по выборке:", df_model['any_purchase_history'].mean().round(4))

  gender age    income  any_purchase_history
0      F  55  112000.0                     1
1      F  75  100000.0                     1
2      M  68   70000.0                     1
3      M  65   53000.0                     1
4      M  58   51000.0                     1

Доля покупок в целом по выборке: 0.9775


In [7]:
# Сохраняем в формате parquet (это как csv, но быстрее и меньше весит)
profile_clean.to_parquet('../data/processed/profile_clean.parquet')
transcript_clean.to_parquet('../data/processed/transcript_clean.parquet')
portfolio.to_parquet('../data/processed/portfolio.parquet')

print("Данные успешно сохранены в папку data/processed!")

Данные успешно сохранены в папку data/processed!
